## Setup

### General Imports

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

### Dataset

Sensitive attributes include:
- sex
- race

Target variable:
- is_recid

In [12]:
PATH_ORG_DATA = "data/compas-scores-two-years.csv"
PATH_SYNTHETIC_DATA = "data/synthetic/"
PATH_FOLDER_MODELS = "models/"

In [13]:
scores_two_years = pd.read_csv(PATH_ORG_DATA)

columns_to_include = [
    "name",
    "first",
    "last",
    "sex",
    "dob",
    "age",
    "age_cat",
    "race",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "priors_count",
    "c_case_number",
    "c_charge_degree",
    "c_charge_desc",
    "r_case_number",
    "r_charge_degree",
    "r_charge_desc",
    "in_custody",
    "out_custody",
    "two_year_recid",
    "is_recid",
]

dataset = scores_two_years[columns_to_include].copy()

n = dataset.shape[0]

In [14]:
categorical_columns = [
    "name",
    "first",
    "last",
    "sex",
    "dob",
    "age_cat",
    "race",
    "c_case_number",
    "c_charge_degree",
    "c_charge_desc",
    "r_case_number",
    "r_charge_degree",
    "r_charge_desc",
    "in_custody",
    "out_custody",
    "two_year_recid",
    "is_recid",
]

## Data Preprocessing

When `is_recid` is 0 there should not be any values in `r_charge_degree` and `r_charge_desc`, as they contain information about the crime they were re-arested for. Thus, we made a separate category "Not applicable" to make it clear which values are truly missing.

In [15]:
recid_charge_cols = [
    "r_case_number",
    "r_charge_degree",
    "r_charge_desc",
]

for col in recid_charge_cols:
    dataset[col] = np.where(
        (dataset["is_recid"] == 0), "Not applicable", dataset[col]
    )

The `race` attribute is distributed as indicated in the table below. The ammount of data on "Hispanic", "Asian", and "Native American" individually is not enough to be significant, therefore we combine these together with "Other".

| Race             | Count | Percent |
| --- | --- | ---  |
| African-American |  3696 |  51.23% |
| Caucasian        |  2454 |  34.02% |
| Hispanic         |   637 |   8.83% |
| Other            |   377 |   5.23% |
| Asian            |    32 |   0.44% |
| Native American  |    18 |   0.25% |

In [16]:
dataset["race"].replace(
    ["Hispanic", "Asian", "Native American"], "Other", inplace=True
)

/var/folders/2m/361556sx6lbc15sc52frn4j40000gn/T/ipykernel_11942/3526782343.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dataset["race"].replace(


## Model Settings

In [17]:
sensitive_features = ["sex", "race"]
target_feature = "is_recid"

In [18]:
verbose = 2
synthetic_data_count = 5000
num_epochs = 200
batch_size = 250

## Model Training

#### Model Imports

In [19]:
from ctgan import CTGAN
from ctgan import TVAE
from tabfairgan import TFG
from fairgan import FairGAN

import pickle

: 

#### CTGAN

The CTGAN model learns the joint distribution of the dataset.

In [20]:
# Train CTGAN model
ctgan_df = dataset.copy()

ctgan_model = CTGAN(epochs=num_epochs, batch_size=batch_size, verbose=verbose)
ctgan_model.fit(ctgan_df, discrete_columns=categorical_columns)

# Generate synthetic data using CTGAN
ctgan_synthetic_data = ctgan_model.sample(synthetic_data_count)
ctgan_synthetic_data.to_csv(
    PATH_SYNTHETIC_DATA + "ctgan_synthetic.csv", index=False
)

# Save the trained CTGAN model
os.makedirs(PATH_FOLDER_MODELS, exist_ok=True)
with open(PATH_FOLDER_MODELS + "ctgan_model.pkl", "wb") as f:
    pickle.dump(ctgan_model, f)

Gen. (0.00) | Discrim. (0.00):   0%|          | 0/200 [00:00<?, ?it/s]

#### TVAE

TVAE learns a latent representation of the data and reconstructs it through a decoder network.

Unlike CTGAN, TVAE optimizes a probabilistic objective rather than using adversarial training.

In [ ]:
# Train TVAE model
tvae_df = dataset.copy()

tvae_model = TVAE(epochs=num_epochs, batch_size=batch_size, verbose=verbose)
tvae_model.fit(tvae_df, discrete_columns=categorical_columns)

# Generate synthetic data using TVAE
tvae_synthetic_data = tvae_model.sample(synthetic_data_count)
tvae_synthetic_data.to_csv(
    PATH_SYNTHETIC_DATA + "tvae_synthetic.csv", index=False
)

with open(PATH_FOLDER_MODELS + "tvae_model.pkl", "wb") as f:
    pickle.dump(tvae_model, f)

#### FairGAN

Our implementation of the FairGAN model can be found [here](fairgan.py)

In [ ]:
# Train FairGAN model
fairgan_df = dataset.copy()


# TODO: Make sure race can be handled
fairgan_model = FairGAN(
    fairgan_df, "sex", target_feature, epochs=num_epochs, verbose=verbose
)
fairgan_model.train()


# Generate synthetic data using FairGAN
fairgan_synthetic_data = fairgan_model.generate(synthetic_data_count)
fairgan_synthetic_data.to_csv(
    PATH_SYNTHETIC_DATA + "fairgan_synthetic.csv", index=False
)

with open(PATH_FOLDER_MODELS + "fairgan_model.pkl", "wb") as f:
    pickle.dump(fairgan_model, f)

#### TabFairGAN

TabFairGAN differs from standard generative models such as CTGAN in that fairness constraints are incorporated directly into the training process.

Instead of using a separate fit function, the model is trained using a custom training loop.

**Fairness Constraints**

To generate fair synthetic data, you need to pass a dictionary containing the following parameters:

- `fair_epochs`: Number of fair training epochs (integer).
- `lamda`: Lambda parameter controlling the trade-off between fairness and accuracy (float).
- `S`: Protected attribute (string, e.g., "sex").
- `Y`: Decision label (string, e.g., "income").
- `S_under`: Value representing the underprivileged group for the protected attribute (string, e.g., " Female").
- `Y_desire`: Desired value for the label (string, e.g., " >50K").

In [ ]:
# TODO: Define proper Y_desire

lambda_values = [0.0, 0.1, 0.5, 0.7, 1.0]

tabfairgan_datasets = {}

for lam in lambda_values:

    fairness_config = {
        "fair_epochs": 50,
        "lamda": lam,
        "S": "race",
        "Y": "is_recid",
        "S_under": "Female",
        "Y_desire": 0,   # or 1 depending on desired outcome
    }

    print(f"\nTraining TabFairGAN with λ = {lam}")

    tfg_df = dataset.copy()

    # Initialize model
    tfg_model = TFG(
        tfg_df,
        epochs=num_epochs,
        batch_size=batch_size,
        fairness_config=fairness_config,
    )

    # Train model
    tfg_model.train()

    # Generate synthetic data
    tfg_synthetic_data = tfg_model.generate_fake_df(
        num_rows=synthetic_data_count
    )

    # Dataset naming
    dataset_name = f"TabFairGAN_λ{lam}"

    # Store dataset
    tabfairgan_datasets[dataset_name] = tfg_synthetic_data

    # Save synthetic dataset
    tfg_synthetic_data.to_csv(
        PATH_SYNTHETIC_DATA + f"{dataset_name}.csv",
        index=False,
    )

    # Save model
    with open(
        PATH_FOLDER_MODELS + f"{dataset_name}_model.pkl",
        "wb",
    ) as f:
        pickle.dump(tfg_model, f)